In [1]:
import gc
gc.collect()

76

In [2]:
import numpy as np
import matplotlib.pyplot as plt
from core.file_manager import preprocess_file_manager
from core.visualization_lib import folder_shower, normalize_volume
from core.helper import register_and_resample, sikit_to_just_data
from core.helper import load_NiFty_and_save_raw_data
from core.stat_calc import find_periods
from core.visualization_lib import show_transformation

ModuleNotFoundError: No module named 'file_manager'

In [ ]:
orginal_data_folder =  '/home/robakp/Exeriments1/prostate_lesion_detection/rjozwiak-MGR_dataset_correct/MGR_dataset_correct'

channels = {
    'adc' : 'adc',
    'anatomy' : 'anatomy',
    'dwi' : 'dwi',
    't2' : 't2'
}

target = 'lesion'

file_extention = '.nii.gz'

preprocessed_steps = [
    'raw',
    'filling_anatomy_gaps'
]

crop_size = (160,160,24)

In [ ]:
file_manager = preprocess_file_manager('/home/robakp/Exeriments1/prostate_lesion_detection/juzwiak_preprocessed',preprocessed_steps,channels)

copy data to preprocess folder

In [ ]:
load_NiFty_and_save_raw_data(orginal_data_folder,file_manager,channels, filter=['3322'])
patients = file_manager.get_file_names()
#for now NO regiter _and_register

Check of Patients 

In [ ]:
# resample_channels = ['adc','dwi'],resample_to = 't2'

# def patient_registration(resample_channels = [],resample_to = 't2'):      
#     for key in resample_channels:
#                 print("ERROR ERROR")
#                 scikit_images[key] = register_and_resample(scikit_images[key],scikit_images[resample_to])  

dispaly

In [ ]:
folder_shower(file_manager,'raw',normalize_volume)

remember about allingning

<h2>Working with holes in prostate layer</h2>

In [ ]:
from anatomy_gap_fixer import find_gaps_in_anatomy
from anatomy_gap_fixer import fix_patient_anatomy

checking out outliers

In [ ]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
for outlier in outliers:
    print(outlier)
    prostate = file_manager.load_file('raw',outlier)['anatomy'] 
    valid_layers = np.any(prostate == 1, axis=(0, 1))
    true_indices = np.where(valid_layers)[0]
    periods = find_periods(true_indices)
    print(len(periods))
    print(periods)

FIX outliers

In [ ]:
outliers, _ = find_gaps_in_anatomy(patients,file_manager)
print(outliers)
start_step = 'raw'
end_step = 'raw'

for outlier in outliers:
    outlier_data = file_manager.load_file(start_step, outlier)
    outlier_new_data = fix_patient_anatomy(outlier_data)
    file_manager.save_file(end_step,outlier,outlier_new_data)

<h2>CROPPING!!!</h2>

Cutting pictures into correct sizes

two ways of centering

finding maximum prostate dimentions

In [ ]:
from stat_calc import find_patients_max_prostate_sizes

step = 'raw'
maximum_prostate_size = find_patients_max_prostate_sizes(patients,file_manager)
print(maximum_prostate_size)

center cropping

In [ ]:
from cropping_lib import center_crop_shift
from stat_calc import find_centroid_non_weighted

THERE IS NO PADDING!!!!

In [ ]:
step = 'raw'

for patient in patients:

    data = file_manager.load_file(step, patient)
    center = find_centroid_non_weighted(data['anatomy'])
    for channel in data:
        data[channel] = center_crop_shift(data[channel],center,crop_size)    
    file_manager.save_file('cropped',patient,data)
    

In [ ]:
folder_shower(file_manager,normalizer=normalize_volume)
folder_shower(file_manager,step = 'cropped',normalizer=normalize_volume)